In [2]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.utils import to_categorical

# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# One-hot encoding
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Model configurations
configs = [
    ('Baseline', None, None),
    ('Xavier', 'glorot_uniform', None),
    ('Kaiming', 'he_normal', None),
    ('Dropout', None, 'dropout'),
    ('L1', None, 'l1'),
    ('L2', None, 'l2')
]

optimizers = ['adam', 'sgd']
results = []

# Train 12 models
for name, init, reg in configs:
    for opt in optimizers:

        model = Sequential([
            Input(shape=(32, 32, 3)),
            Flatten()
        ])

        for units in [256, 128]:
            kwargs = {'activation': 'relu'}

            if init:
                kwargs['kernel_initializer'] = init

            if reg == 'l1':
                kwargs['kernel_regularizer'] = l1(0.001)
            elif reg == 'l2':
                kwargs['kernel_regularizer'] = l2(0.001)

            model.add(Dense(units, **kwargs))

            if reg == 'dropout':
                model.add(Dropout(0.5))

        model.add(Dense(10, activation='softmax'))

        model.compile(
            optimizer=opt,
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        print(f"\nTraining {name} with {opt}")

        model.fit(
            x_train, y_train,
            epochs=5,
            batch_size=128,
            verbose=1
        )

        loss, acc = model.evaluate(x_test, y_test, verbose=0)

        results.append([
            name, reg if reg else 'None',
            opt.upper(), round(acc * 100, 2),
            round(loss, 4)
        ])

# Comparison table
df = pd.DataFrame(results, columns=[
    'Model', 'Regularization', 'Optimizer',
    'Test Accuracy (%)', 'Test Loss'
])

print("\nMODEL PERFORMANCE COMPARISON")
print(df.to_string(index=False))


Training Baseline with adam
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 23ms/step - accuracy: 0.3168 - loss: 1.9042
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.3921 - loss: 1.7006
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.4228 - loss: 1.6184
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.4468 - loss: 1.5552
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.4601 - loss: 1.5200

Training Baseline with sgd
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.2842 - loss: 2.0080
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.3512 - loss: 1.8330
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.3774 - loss: 1.7657
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.3958 - loss: 1.7197
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.4103 - loss: 1.6778

Training Xavier with adam
Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step